In [40]:
from pathlib import Path
import pandas as pd
import numpy as np

# go 2 levels up to repo root, then into data
statsbomb_dir = Path("../../data/raw/Statsbomb")

events = pd.read_parquet(statsbomb_dir / "events.parquet")
matches = pd.read_parquet(statsbomb_dir / "matches.parquet")
lineups = pd.read_parquet(statsbomb_dir / "lineups.parquet")

In [7]:
# Shot events
shots = events.loc[events["type"] == "Shot"].copy()

# Team xG per match
team_match_xg = (
    shots.groupby(["match_id", "team"], as_index=False)
    .agg(
        team_xg=("shot_statsbomb_xg", "sum"),
        shots=("id", "count")
    )
)

# Match outcomes
match_outcomes = matches[
    ["match_id", "home_team", "away_team", "home_score", "away_score"]
].copy()

home_outcomes = match_outcomes.assign(
    team=match_outcomes["home_team"],
    opponent=match_outcomes["away_team"],
    goals_for=match_outcomes["home_score"],
    goals_against=match_outcomes["away_score"]
)

away_outcomes = match_outcomes.assign(
    team=match_outcomes["away_team"],
    opponent=match_outcomes["home_team"],
    goals_for=match_outcomes["away_score"],
    goals_against=match_outcomes["home_score"]
)

team_outcomes = pd.concat([home_outcomes, away_outcomes], ignore_index=True)

team_outcomes["outcome"] = np.where(
    team_outcomes["goals_for"] > team_outcomes["goals_against"], "Win",
    np.where(
        team_outcomes["goals_for"] < team_outcomes["goals_against"], "Loss",
        "Draw"
    )
)

team_outcomes = team_outcomes[["match_id", "team", "opponent", "outcome"]]

# Merge outcomes with xG
team_match = team_match_xg.merge(
    team_outcomes,
    on=["match_id", "team"],
    how="left"
)

# Add opponent xG
opp_xg = team_match_xg.rename(
    columns={"team": "opponent", "team_xg": "opp_xg"}
)

team_match = team_match.merge(
    opp_xg[["match_id", "opponent", "opp_xg"]],
    on=["match_id", "opponent"],
    how="left"
)

# xG difference
team_match["xg_diff"] = (team_match["team_xg"] - team_match["opp_xg"]).abs()

XG_PARITY_THRESHOLD = 0.3

xg_parity = team_match.loc[
    team_match["xg_diff"] <= XG_PARITY_THRESHOLD
].copy()

xg_parity = xg_parity[
    [
        "match_id",
        "team",
        "opponent",
        "team_xg",
        "opp_xg",
        "xg_diff",
        "shots",
        "outcome"
    ]
].sort_values(["match_id", "team"]).reset_index(drop=True)

display(xg_parity)

,match_id,team,opponent,team_xg,opp_xg,xg_diff,shots,outcome
0,7480,OL Reign,Utah Royals,0.864332,0.861828,0.002504,9,Win
1,7480,Utah Royals,OL Reign,0.861828,0.864332,0.002504,11,Loss
2,7494,Houston Dash,Utah Royals,1.131186,1.236244,0.105058,16,Loss
3,7494,Utah Royals,Houston Dash,1.236244,1.131186,0.105058,9,Win
4,7497,Chicago Red Stars,North Carolina Courage,0.940007,1.144564,0.204556,12,Draw
...,...,...,...,...,...,...,...,...
1297,3998854,Portugal Women's,Belgium Women's,2.010840,2.062541,0.051702,20,Loss
1298,4018354,Italy Women's,Norway Women's,1.597245,1.895418,0.298173,14,Win
1299,4018354,Norway Women's,Italy Women's,1.895418,1.597245,0.298173,10,Loss
1300,4018355,England Women's,Sweden Women's,7.395617,7.213724,0.181893,25,Draw


In [8]:
xg_parity.duplicated(subset=["match_id", "team"]).sum()

np.int64(0)

In [ ]:
team_level = xg_parity[["match_id", "team", "outcome"]].copy()
team_level.duplicated(subset=["match_id", "team"]).sum()


np.int64(0)

In [30]:
team_level.shape

(1302, 3)

In [16]:
progressive_events = events[events["match_id"].isin(xg_parity["match_id"])].copy()
progressive_events.head()

,id,index_num,period,minute,second,timestamp,duration,location_x,location_y,possession,...,block_save_block,ball_recovery_offensive,ball_recovery_failure,miscontrol_aerial_won,substitution_replacement_id,substitution_replacement_name,substitution_outcome,fifty_fifty_outcome,bad_behaviour_card,injury_stoppage_in_chain
37368,5e3d09b4-4741-43af-b3ca-9b451c458b3a,1,1,0,0,00:00:00,0.000000,NaN,NaN,1,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37369,f2c941ff-922b-4296-a1b9-a494f394229b,2,1,0,0,00:00:00,0.000000,NaN,NaN,1,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37370,4bbe1ded-478b-4817-990f-80a5868c7f19,3,1,0,0,00:00:00,0.000000,NaN,NaN,1,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37371,8162edda-6847-40e8-bebe-20f408c9d9d0,4,1,0,0,00:00:00,0.000000,NaN,NaN,1,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37372,bb966c93-fe22-48c9-bd9d-64435865e94f,5,1,0,0,00:00:00.684,0.829732,60.0,40.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False


In [18]:
prog_events = progressive_events[progressive_events["type"].isin(["Pass", "Carry"])].copy()
prog_events.head()

,id,index_num,period,minute,second,timestamp,duration,location_x,location_y,possession,...,block_save_block,ball_recovery_offensive,ball_recovery_failure,miscontrol_aerial_won,substitution_replacement_id,substitution_replacement_name,substitution_outcome,fifty_fifty_outcome,bad_behaviour_card,injury_stoppage_in_chain
37372,bb966c93-fe22-48c9-bd9d-64435865e94f,5,1,0,0,00:00:00.684,0.829732,60.0,40.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37374,ce36226b-04fb-4b7b-bb8e-daed162a0327,7,1,0,1,00:00:01.513,2.900568,48.0,41.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37375,f918de85-0a24-4e3d-8e57-6ebca67df541,8,1,0,4,00:00:04.414,0.612700,54.0,40.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37377,a2665779-fa3f-4024-8f74-721003e4bacb,10,1,0,5,00:00:05.027,1.010446,52.0,32.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37379,9964ac94-826e-478c-8c61-a579ee744bdc,12,1,0,6,00:00:06.037,2.025454,52.0,39.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False


In [20]:
prog_events["is_progressive"] = (
    (prog_events["type"] == "Pass") &
    (prog_events["pass_end_location_x"] - prog_events["location_x"] >= 10)
) | (
    (prog_events["type"] == "Carry") &
    (prog_events["carry_end_location_x"] - prog_events["location_x"] >= 10)
)
prog_events.head()

,id,index_num,period,minute,second,timestamp,duration,location_x,location_y,possession,...,ball_recovery_offensive,ball_recovery_failure,miscontrol_aerial_won,substitution_replacement_id,substitution_replacement_name,substitution_outcome,fifty_fifty_outcome,bad_behaviour_card,injury_stoppage_in_chain,is_progressive
37372,bb966c93-fe22-48c9-bd9d-64435865e94f,5,1,0,0,00:00:00.684,0.829732,60.0,40.0,2,...,False,False,False,NaN,NaN,NaN,NaN,NaN,False,False
37374,ce36226b-04fb-4b7b-bb8e-daed162a0327,7,1,0,1,00:00:01.513,2.900568,48.0,41.0,2,...,False,False,False,NaN,NaN,NaN,NaN,NaN,False,False
37375,f918de85-0a24-4e3d-8e57-6ebca67df541,8,1,0,4,00:00:04.414,0.612700,54.0,40.0,2,...,False,False,False,NaN,NaN,NaN,NaN,NaN,False,False
37377,a2665779-fa3f-4024-8f74-721003e4bacb,10,1,0,5,00:00:05.027,1.010446,52.0,32.0,2,...,False,False,False,NaN,NaN,NaN,NaN,NaN,False,False
37379,9964ac94-826e-478c-8c61-a579ee744bdc,12,1,0,6,00:00:06.037,2.025454,52.0,39.0,2,...,False,False,False,NaN,NaN,NaN,NaN,NaN,False,False


In [22]:
progressive_actions = prog_events[prog_events["is_progressive"]].copy()
progressive_actions.shape

(289731, 113)

In [23]:
def classify_zone(y):
    return "Wide" if (y < 20 or y > 60) else "Central"

progressive_actions["zone"] = progressive_actions["location_y"].apply(classify_zone)
progressive_actions.head()

,id,index_num,period,minute,second,timestamp,duration,location_x,location_y,possession,...,ball_recovery_failure,miscontrol_aerial_won,substitution_replacement_id,substitution_replacement_name,substitution_outcome,fifty_fifty_outcome,bad_behaviour_card,injury_stoppage_in_chain,is_progressive,zone
37392,290d4fbd-e375-4649-8372-331eb679c0e8,25,1,0,24,00:00:24.078,1.145729,57.0,22.0,2,...,False,False,NaN,NaN,NaN,NaN,NaN,False,True,Central
37395,603e9051-f3f2-47ba-abe7-dd4ebe1558d6,28,1,0,26,00:00:26.283,1.017700,72.0,5.0,2,...,False,False,NaN,NaN,NaN,NaN,NaN,False,True,Wide
37411,15f34cbb-e3b7-45bd-83d5-d4ca863dba60,44,1,0,38,00:00:38.642,1.868790,54.0,42.0,2,...,False,False,NaN,NaN,NaN,NaN,NaN,False,True,Central
37422,1da68b22-2ce9-4c0d-9eb8-a8c638bba4a8,55,1,1,3,00:01:03.004,1.766034,84.0,18.0,4,...,False,False,NaN,NaN,NaN,NaN,NaN,False,True,Wide
37435,f5b2c668-ede0-4636-961d-9fdbce490576,68,1,1,13,00:01:13.846,2.738162,72.0,52.0,4,...,False,False,NaN,NaN,NaN,NaN,NaN,False,True,Central


In [24]:
progressive_actions.shape

(289731, 114)

In [25]:
team_prog_total = (
    progressive_actions
    .groupby(["match_id", "team"])
    .size()
    .reset_index(name="team_progressive_actions")
)

team_prog_total

,match_id,team,team_progressive_actions
0,7480,OL Reign,228
1,7480,Utah Royals,226
2,7494,Houston Dash,213
3,7494,Utah Royals,301
4,7497,Chicago Red Stars,235
...,...,...,...
1297,3998854,Portugal Women's,201
1298,4018354,Italy Women's,189
1299,4018354,Norway Women's,167
1300,4018355,England Women's,292


In [27]:
team_wide_prog = (
    progressive_actions[progressive_actions["zone"] == "Wide"]
    .groupby(["match_id", "team"])
    .size()
    .reset_index(name="wide_progressive_actions")
)
team_wide_prog

,match_id,team,wide_progressive_actions
0,7480,OL Reign,112
1,7480,Utah Royals,97
2,7494,Houston Dash,94
3,7494,Utah Royals,109
4,7497,Chicago Red Stars,89
...,...,...,...
1297,3998854,Portugal Women's,98
1298,4018354,Italy Women's,79
1299,4018354,Norway Women's,96
1300,4018355,England Women's,159


In [28]:
team_width = team_prog_total.merge(
    team_wide_prog,
    on=["match_id", "team"],
    how="left"
).fillna(0)
team_width

,match_id,team,team_progressive_actions,wide_progressive_actions
0,7480,OL Reign,228,112
1,7480,Utah Royals,226,97
2,7494,Houston Dash,213,94
3,7494,Utah Royals,301,109
4,7497,Chicago Red Stars,235,89
...,...,...,...,...
1297,3998854,Portugal Women's,201,98
1298,4018354,Italy Women's,189,79
1299,4018354,Norway Women's,167,96
1300,4018355,England Women's,292,159


In [29]:
team_width["wide_progression_share"] = (
    team_width["wide_progressive_actions"] /
    team_width["team_progressive_actions"]
)
team_width

,match_id,team,team_progressive_actions,wide_progressive_actions,wide_progression_share
0,7480,OL Reign,228,112,0.491228
1,7480,Utah Royals,226,97,0.429204
2,7494,Houston Dash,213,94,0.441315
3,7494,Utah Royals,301,109,0.362126
4,7497,Chicago Red Stars,235,89,0.378723
...,...,...,...,...,...
1297,3998854,Portugal Women's,201,98,0.487562
1298,4018354,Italy Women's,189,79,0.417989
1299,4018354,Norway Women's,167,96,0.574850
1300,4018355,England Women's,292,159,0.544521


In [31]:
team_level = team_level.merge(
    team_width[["match_id", "team", "wide_progression_share"]],
    on=["match_id", "team"],
    how="left"
)
team_level

,match_id,team,outcome,wide_progression_share
0,7480,OL Reign,Win,0.491228
1,7480,Utah Royals,Loss,0.429204
2,7494,Houston Dash,Loss,0.441315
3,7494,Utah Royals,Win,0.362126
4,7497,Chicago Red Stars,Draw,0.378723
...,...,...,...,...
1297,3998854,Portugal Women's,Loss,0.487562
1298,4018354,Italy Women's,Win,0.417989
1299,4018354,Norway Women's,Loss,0.574850
1300,4018355,England Women's,Draw,0.544521


In [32]:
TOUCH_EVENTS = [
    "Ball Receipt*", 
    "Pass", 
    "Carry", 
    "Dribble", 
    "Shot"
]

In [33]:
touches = events[
    (events["match_id"].isin(xg_parity["match_id"])) &
    (events["type"].isin(TOUCH_EVENTS))
].copy()
touches.head()

,id,index_num,period,minute,second,timestamp,duration,location_x,location_y,possession,...,block_save_block,ball_recovery_offensive,ball_recovery_failure,miscontrol_aerial_won,substitution_replacement_id,substitution_replacement_name,substitution_outcome,fifty_fifty_outcome,bad_behaviour_card,injury_stoppage_in_chain
37372,bb966c93-fe22-48c9-bd9d-64435865e94f,5,1,0,0,00:00:00.684,0.829732,60.0,40.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37373,2dfc30a1-f20a-4068-8035-4a6d87d5eaaa,6,1,0,1,00:00:01.513,NaN,48.0,41.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37374,ce36226b-04fb-4b7b-bb8e-daed162a0327,7,1,0,1,00:00:01.513,2.900568,48.0,41.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37375,f918de85-0a24-4e3d-8e57-6ebca67df541,8,1,0,4,00:00:04.414,0.612700,54.0,40.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False
37376,a50140c3-8bd5-44ae-a3ca-3d1373c13f32,9,1,0,5,00:00:05.027,NaN,52.0,32.0,2,...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,False


In [34]:
touches.shape

(1736680, 112)

In [35]:
POSITION_BIN_MAP = {
    "Center Back": "Center Back",
    "Left Center Back": "Center Back",
    "Right Center Back": "Center Back",

    "Left Back": "Fullback",
    "Right Back": "Fullback",
    "Left Wing Back": "Fullback",
    "Right Wing Back": "Fullback",

    "Center Defensive Midfield": "Defensive Midfield",
    "Left Defensive Midfield": "Defensive Midfield",
    "Right Defensive Midfield": "Defensive Midfield",

    "Center Midfield": "Central Midfield",
    "Left Center Midfield": "Central Midfield",
    "Right Center Midfield": "Central Midfield",

    "Center Attacking Midfield": "Attacking Midfield",
    "Left Attacking Midfield": "Attacking Midfield",
    "Right Attacking Midfield": "Attacking Midfield",

    "Left Wing": "Wide Forward",
    "Right Wing": "Wide Forward",
    "Left Midfield": "Wide Forward",
    "Right Midfield": "Wide Forward",

    "Center Forward": "Striker",
    "Left Center Forward": "Striker",
    "Right Center Forward": "Striker",
    "Secondary Striker": "Striker",

    "Goalkeeper": "Goalkeeper",
}

In [36]:
touches["position_bin"] = touches["position"].map(POSITION_BIN_MAP)
touches = touches.dropna(subset=["position_bin"])

In [37]:
touches.shape

(1736680, 113)

In [38]:
team_pos_touches = (
    touches
    .groupby(["match_id", "team", "position_bin"])
    .size()
    .reset_index(name="touches")
)

team_pos_touches.head()

,match_id,team,position_bin,touches
0,7480,OL Reign,Center Back,222
1,7480,OL Reign,Central Midfield,357
2,7480,OL Reign,Fullback,265
3,7480,OL Reign,Goalkeeper,60
4,7480,OL Reign,Striker,102


In [39]:
team_pos_touches.shape

(9039, 4)

In [41]:
BUILD_UP = ["Center Back", "Fullback", "Defensive Midfield"]

team_pos_touches["phase"] = np.where(
    team_pos_touches["position_bin"].isin(BUILD_UP),
    "BuildUp",
    "Advanced"
)

team_pos_touches

,match_id,team,position_bin,touches,phase
0,7480,OL Reign,Center Back,222,BuildUp
1,7480,OL Reign,Central Midfield,357,Advanced
2,7480,OL Reign,Fullback,265,BuildUp
3,7480,OL Reign,Goalkeeper,60,Advanced
4,7480,OL Reign,Striker,102,Advanced
...,...,...,...,...,...
9034,4018355,Sweden Women's,Defensive Midfield,206,BuildUp
9035,4018355,Sweden Women's,Fullback,205,BuildUp
9036,4018355,Sweden Women's,Goalkeeper,74,Advanced
9037,4018355,Sweden Women's,Striker,130,Advanced


In [42]:
team_phase = (
    team_pos_touches
    .groupby(["match_id", "team", "phase"])["touches"]
    .sum()
    .reset_index()
)

team_phase

,match_id,team,phase,touches
0,7480,OL Reign,Advanced,739
1,7480,OL Reign,BuildUp,487
2,7480,Utah Royals,Advanced,436
3,7480,Utah Royals,BuildUp,669
4,7494,Houston Dash,Advanced,621
...,...,...,...,...
2599,4018354,Norway Women's,BuildUp,569
2600,4018355,England Women's,Advanced,719
2601,4018355,England Women's,BuildUp,937
2602,4018355,Sweden Women's,Advanced,545


In [43]:
team_phase = team_phase.pivot(
    index=["match_id", "team"],
    columns="phase",
    values="touches"
).reset_index().fillna(0)
team_phase

phase,match_id,team,Advanced,BuildUp
0,7480,OL Reign,739,487
1,7480,Utah Royals,436,669
2,7494,Houston Dash,621,452
3,7494,Utah Royals,866,938
4,7497,Chicago Red Stars,889,321
...,...,...,...,...
1297,3998854,Portugal Women's,559,754
1298,4018354,Italy Women's,565,618
1299,4018354,Norway Women's,530,569
1300,4018355,England Women's,719,937


In [44]:
team_phase["build_up_share"] = (
    team_phase["BuildUp"] /
    (team_phase["BuildUp"] + team_phase["Advanced"])
)

team_phase

phase,match_id,team,Advanced,BuildUp,build_up_share
0,7480,OL Reign,739,487,0.397227
1,7480,Utah Royals,436,669,0.605430
2,7494,Houston Dash,621,452,0.421249
3,7494,Utah Royals,866,938,0.519956
4,7497,Chicago Red Stars,889,321,0.265289
...,...,...,...,...,...
1297,3998854,Portugal Women's,559,754,0.574257
1298,4018354,Italy Women's,565,618,0.522401
1299,4018354,Norway Women's,530,569,0.517743
1300,4018355,England Women's,719,937,0.565821


In [45]:
team_level = team_level.merge(
    team_phase[["match_id", "team", "build_up_share"]],
    on=["match_id", "team"],
    how="left"
)

team_level

,match_id,team,outcome,wide_progression_share,build_up_share
0,7480,OL Reign,Win,0.491228,0.397227
1,7480,Utah Royals,Loss,0.429204,0.605430
2,7494,Houston Dash,Loss,0.441315,0.421249
3,7494,Utah Royals,Win,0.362126,0.519956
4,7497,Chicago Red Stars,Draw,0.378723,0.265289
...,...,...,...,...,...
1297,3998854,Portugal Women's,Loss,0.487562,0.574257
1298,4018354,Italy Women's,Win,0.417989,0.522401
1299,4018354,Norway Women's,Loss,0.574850,0.517743
1300,4018355,England Women's,Draw,0.544521,0.565821
